# VM SC — Square critic α cap: `square_tent12i_cap03` × 3 + `square_tent12i_cap1` × 3 + Can 회귀 `can_tent12i_cap03` × 3 (vm3_new 판형, 0→9 순서)

전용 G4 VM에서 **0부터 9까지 순서대로**. 1번은 런타임 재시작 → 2번부터. **코드 변경 있음**(`train.critic_alpha_cap`, 9/15 — 6번 셀의 단위 테스트와 cap 스모크가 통과해야 7번), 9 run(Square 6 + Can 3, RAM ≈ 155 GB, 5~6시간), 9번이 끝나면 VM 반납. 기준 문서: `HANDOFF.md` §24. 짝 노트북: `colab/vm_square_rs.ipynb`(갈래 A 보상 정규화 + 대조군 `square_tent12i`). 두 VM을 같은 밤에 띄운다.

**배경(§19.8·§19.11·§20.5, `results/2026-09-08/README.md` §6)**: Square에서 목표 엔트로피 12(`square_tent12`)는 42k dip을 없애지만(env 42,032 seed 평균 0.404 vs baseline 0.234) 후반에 무너진다(127k 0.369 < baseline 0.407 < π_dp 0.494). 진단(9/8 `diagnostic_groups.csv`): 엔트로피 12를 지키느라 α가 online 0–5k 0.40 → 10k 0.62 → 20k 1.3 → 50k 6.4 → 110k 20까지 오르고, critic 타깃의 보너스 −α·log π′가 γ = 0.999로 쌓여 `qw_mean`이 143 → 30,916. hard backup(`square_tent12_hq`, β = 0)은 후반을 살리지만(127k 0.490; tent12 s1–3 대비 +0.093 ± 0.020, 3/3) 초반을 잃는다(초기 AUC −0.050 ± 0.040, 42k 0.330). hq에서는 α가 0.17–0.37에 머문다 — 즉 α 폭주는 보너스가 Q를 부풀리는 되먹임의 결과다. Can(γ 0.99)에서는 β = 0이 해롭고(§22 `can_tent12i_hq`, 전 구간 −0.2) 보너스가 필요는 하다 — **폭주만 막아야** 한다.

## 갈래 B 질문: critic 타깃의 보너스에만 상한 α_c = min(α, cap)을 걸면(actor의 α는 auto 그대로) hq(β = 0)의 초반 비용 없이 후반 붕괴를 막는가
- 구현(`o2o_utils.critic_ent_coef`): 타깃 `r + γ(Q̄ − min(α, cap)·log π′)`, actor 손실 `α·log π − Q_W`는 그대로. train_log 새 열 `critic_ent_coef`(타깃이 실제 쓴 온도; cap이 물리면 = cap, 아니면 = `ent_coef`). fingerprint에 `critic_alpha_cap` 포함(다른 cap의 checkpoint는 resume 거부). cap ≤ 0 = 끔(기본 −1, 상류와 같은 연산).
- `square_tent12i_cap03_s{1,2,3}`: `train.ent_coef=auto_0.3 train.target_ent=12 train.critic_alpha_cap=0.3`. tent12의 α 궤적이면 cap 0.3은 online ~5–10k(env 37–42k)부터 끝까지 물린다 — 42k 시점의 critic 온도는 tent12 ≈ 0.6 / cap03 0.3 / hq 0. 보너스 상한 0.3 × 12 ≈ 3.6/step → Q_W 상한 ≈ Σγ^t(에피소드 100 청크) × 3.6 + 수익 ≈ 수백(tent12 23,000).
- `square_tent12i_cap1_s{1,2,3}`: `… train.critic_alpha_cap=1.0` — 용량 arm. tent12 궤적이면 online ~15–20k(env 47–52k)부터 물린다. cap03 ≈ cap1이면 "상한 값은 무관, 폭주만 막으면 됨"; cap1이 tent12형 붕괴면 보너스 ~12/step도 이미 과함(→ 상한은 0.3 쪽).
- `can_tent12i_cap03_s{1,2,3}` (회귀): Can tent12i의 α는 online 20k–85k에 0.31–0.42(bin 평균; 봉우리 0.42@45k, seed별 최대 ≈ 0.49)이므로 **cap 0.3은 Can에서도 그 구간에 물린다(no-op이 아니다)** — 보너스를 최대 ~30% 깎는 약한 개입. 이 arm은 "구성상 무해"가 아니라 **"Square를 고치는 같은 처방(cap 0.3)이 Can을 해치지 않는가"**를 잰다. 구성상 no-op은 cap ≥ 0.5에서만 성립하고, 코드 경로의 무해성은 6번 셀의 스모크(cap −1 ⇒ `critic_ent_coef` = `ent_coef`)와 단위 테스트가 잰다.
- 예측: cap03·cap1 모두 actor α가 전 구간 1 아래에 머문다(hq 0.17–0.37처럼 — α 폭주가 보너스 → Q 부풀림 → α의 되먹임이라면 상한이 고리를 끊는다), `qw_mean` < 500, 42k ≥ 0.404, 127k ≥ 0.472. 반증: cap을 걸어도 α가 15–20으로 가면 α 상승은 보너스 되먹임이 아니라 다른 원인(보상 Q 기울기 자체의 성장).

## 사전 판정 (정의는 `results/2026-09-08/README.md` §2: online = env − 32,016; 초기 창 online 0–67,984; 최저는 5k 격자 online 5,008·k의 seed 평균곡선; 후반 env 127,136·200 ep)
- **주 지표 1 = env 42,032(online 10,016) seed 평균**. 참조: π_dp 0.494, iql 0.503, tent12 0.404(n=5; s1–3 0.407), fixalpha_03 0.358, tent12_hq 0.330, baseline 0.234(s1–3 0.210).
- **주 지표 2 = env 127,136 seed 평균**. 참조: mix_prefill 0.622, iql 0.562, π_dp 0.494, tent12_hq 0.490, baseline 0.407(s1–3 0.472), tent12 0.369(s1–3 0.397).
- **관건 = hq형 초반 비용의 회피**: 초기 AUC seed-matched(vs tent12 s1–3) 차이. hq는 −0.050 ± 0.040. cap arm이 ≥ 0(또는 3/3 음수가 아님)이면 "보너스를 남긴 채 폭주만 막음"이 초반을 지킨 것.
- 보조: 초기 창 평균곡선 최저(tent12 0.326@online 50k, hq 0.287@15k, baseline 0.234@10k), 초기 AUC(tent12 0.404, baseline 0.401, hq 0.378), 102k/127k 평균(tent12 s1–3 0.367, baseline s1–3 0.455, hq 0.463, mix_prefill 0.624), seed-matched 차이 vs tent12 s1–3 · tent12_hq s1–3 · baseline s1–3 · (있으면) square_tent12i s1–3(VM SR).
- 진단(판정 아님): `ent_coef`(actor α — 1 아래에 머무는지), `critic_ent_coef`(cap이 물린 구간; = cap이면 물린 것), `qw_mean`(< 500?), `logp_mean`(≈ −12), `mu_absmean`. `q_start − mc_return`은 α_c가 작으므로 후반에 읽을 수 있다.

| 결과 | 해석 |
|---|---|
| 42k ≥ 0.494 **그리고** 127k ≥ 0.62 | 전이 완성 — cap이 두 과제 공통 처방(Can 회귀가 무해할 때) |
| 42k ≥ 0.404(tent12) 그리고 127k ≥ 0.472(baseline s1–3) | 후반 손실만 해결, 초반은 tent12 동급(π_dp 아래 dip은 남음) |
| 127k ≥ 0.472인데 초기 AUC가 tent12 대비 3/3 음수(hq형) | 보너스가 초반에 필요 → cap 값 조정(cap1 결과로 방향) |
| 둘 다 아님 | Q 폭주는 결과지 원인이 아님 → γ 0.999·과제 난이도 쪽 |

- **Can 회귀 `can_tent12i_cap03` 판정**: seed-matched(s1–3) vs `can_tent12i` — online 5,008 평균 ≥ 0.405(참조 tent12i 0.530 ± 0.230), 초기 AUC 차이·129k 차이가 각각 0 ± SE 안이고 3/3 같은 방향이 아니면 **"해치지 않음"**(참조 tent12i 초기 AUC 0.594 ± 0.052, 129k 0.697 ± 0.058, 104k/129k 평균 0.676 ± 0.072). 3/3 같은 방향 + |차이| > SE면 "cap 0.3은 Can에 비용" → Can no-op 보장값 cap 0.5로 재시험. tent12i_hq(전 구간 −0.2)형 손실은 cap에서 나오면 안 된다(보너스 70%는 남으므로).
- 검정력(§21): Square 127k per-seed SD ≈ 0.13 → n=3 SE ≈ 0.075; 2 SE 이상 떨어진 참조(tent12 0.369 vs mix_prefill 0.622)만 가른다. Can 초기 AUC SD 0.035–0.06이라 n=3으로 ±0.05 차이는 읽힌다.
- 갈래 A(VM SR)와의 관계: A가 되면 "스케일 문제였다"가 확정되고 B는 원리적 해법. A가 안 되고 B만 되면 "스케일이 아니라 보너스 누적 자체가 문제".
- 문헌: `HANDOFF.md` §24 '문헌' 항(9/15 검색·검증). 선행 사례가 있으면 그 이름으로 부르고 대조군에 넣는다; 없으면 "우리가 아는 한".
- 쓰면 안 되는 문장: 9/8 README §4·§8, §22–23 목록 그대로. 추가로 "cap이 Can에서 no-op"(20k–85k에 물린다), "Q_W가 폭주해서 후반이 무너졌다"(이 실험이 그 검정), "hq는 baseline보다 나쁘다"(seed-matched 동급).

## 0. Drive 마운트와 GPU 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Conda 설치 — 실행하면 런타임 자동 재시작

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. 재시작 후 Drive 재마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/dsrl_project'
print(PROJ)

## 3. 최신 o2o 저장소 준비 — HEAD가 `critic_alpha_cap` 커밋(9/15) 이상이어야 한다

In [ ]:
%%bash
set -e
git config --global url."https://github.com/".insteadOf "git@github.com:"
if [ -d /content/dsrl/.git ]; then
  git -C /content/dsrl checkout o2o
  git -C /content/dsrl pull --ff-only origin o2o
else
  test ! -e /content/dsrl || { echo '/content/dsrl exists but is not a git checkout'; exit 2; }
  git clone --recurse-submodules -b o2o https://github.com/msp0617/dsrl.git /content/dsrl
fi
git -C /content/dsrl submodule sync --recursive
git -C /content/dsrl submodule update --init --recursive
echo -n 'HEAD: '; git -C /content/dsrl rev-parse --short HEAD
grep -q 'critic_alpha_cap' /content/dsrl/o2o_utils.py || { echo 'o2o_utils.py has no critic_alpha_cap: the 9/15 commit is not on origin/o2o yet'; exit 2; }

## 4. 캐시에서 conda 환경 복원

In [ ]:
%%bash
set -e
CACHE=/content/drive/MyDrive/dsrl_project/env_cache/dsrl_env.tar.gz
test -s "$CACHE" || { echo "missing $CACHE"; exit 2; }
mkdir -p /usr/local/envs
rm -rf /usr/local/envs/dsrl
tar -xzf "$CACHE" -C /usr/local/envs
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python - <<'PY'
import torch, robomimic, robosuite, mujoco, stable_baselines3
assert torch.cuda.is_available(), 'GPU runtime required'
print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))
PY
python /usr/local/envs/dsrl/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py

## 5. Can·Square 정책·정규화 복원과 환경 패치 (이 VM은 두 과제를 다 돌린다)

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
RUNTIME=/content/dsrl/dppo/log
DRIVE=$PROJ/dppo_log
mkdir -p "$RUNTIME"
test -d "$DRIVE" || { echo "missing $DRIVE"; exit 2; }
cp -r "$DRIVE"/. "$RUNTIME"/
restore () {
  REL=$1
  SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$REL" -print -quit)
  test -n "$SRC" || { echo "missing asset: $REL"; exit 2; }
  mkdir -p "$RUNTIME/$(dirname "$REL")"
  [ "$SRC" = "$RUNTIME/$REL" ] || cp -f "$SRC" "$RUNTIME/$REL"
  ls -lh "$RUNTIME/$REL"
}
restore robomimic-pretrain/can/can_pre_diffusion_mlp_ta4_td20/2024-06-28_13-29-54/checkpoint/state_5000.pt
restore robomimic/can/normalization.npz
restore robomimic-pretrain/square/square_pre_diffusion_mlp_ta4_td100_ddim-100steps/2025-04-11_19-13-26_44/checkpoint/state_3000.pt
restore robomimic/square/normalization.npz
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS
python /content/dsrl/colab/patch_env.py

## 6. 사전검사 — 단위 테스트 3종 + cap 스모크(Can, 작은 망, 약 5분) + 이번 exp_id 9개·비교군·RAM

스모크: 고정 α 1.0으로 1,200 env step을 두 번 돌려 `critic_ent_coef` 열이 cap 0.3이면 전부 0.3, cap −1이면 `ent_coef`와 같은지 확인한다(둘 중 하나라도 어긋나면 셀이 실패하고 7번으로 가지 않는다). 스모크 폴더는 확인 뒤 지운다.

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
echo '== 단위 테스트 (torch 불필요): cap 기본값·fingerprint·min(alpha, cap) 산술, resume 상태, launch 문자열'
python scripts/test_offline_mix.py 2>&1 | tail -n 2; test ${PIPESTATUS[0]} -eq 0 || { echo 'test_offline_mix FAILED'; exit 2; }
python scripts/test_resume_state.py 2>&1 | tail -n 2; test ${PIPESTATUS[0]} -eq 0 || { echo 'test_resume_state FAILED'; exit 2; }
python scripts/test_notebook_launches.py 2>&1 | tail -n 2; test ${PIPESTATUS[0]} -eq 0 || { echo 'test_notebook_launches FAILED'; exit 2; }
echo '== cap 스모크 (Can, 1,200 env step x 2)'
CFG_CAN='--config-path=cfg/robomimic --config-name=dsrl_can.yaml'
SMOKE="log_dir=$PROJ/logs env.n_envs=1 env.n_eval_envs=1 num_evals=1 eval_schedule.every_env_early=400 eval_schedule.early_until_env=100000 eval_schedule.num_evals_early=1 ckpt_every_env_steps=400 train.init_rollout_steps=50 train.utd=1 train.noise_critic_grad_steps=1 train.batch_size=32 train.layer_size=256 train.num_layers=2 train.buffer_size=20000 train.total_env_steps=1200 resume=False train.ent_coef=1.0"
for CAP in 0.3 -1; do
  rm -rf "$PROJ/logs/smoke_cap$CAP"
  python train_dsrl.py $CFG_CAN exp_id=smoke_cap$CAP $SMOKE train.critic_alpha_cap=$CAP 2>&1 | grep '\[done\]\|Error\|Traceback' | tail -n 2; test ${PIPESTATUS[0]} -eq 0 || { echo "smoke cap=$CAP FAILED"; exit 2; }
  python - "$PROJ/logs/smoke_cap$CAP/train_log.csv" "$CAP" <<'PY'
import csv, sys
path, cap = sys.argv[1], float(sys.argv[2])
rows = list(csv.DictReader(open(path)))
assert rows and 'critic_ent_coef' in rows[0], 'critic_ent_coef column missing: ' + path
a = [float(r['ent_coef']) for r in rows]
c = [float(r['critic_ent_coef']) for r in rows]
assert all(abs(x - 1.0) < 1e-6 for x in a), 'the actor alpha must stay at the fixed 1.0'
want = [min(x, cap) if cap > 0 else x for x in a]
assert all(abs(x - y) < 1e-6 for x, y in zip(c, want)), f'critic_ent_coef {c[:3]} != expected {want[:3]}'
print(f'OK cap={cap}: {len(rows)} rows, critic_ent_coef {c[0]:.3f} (ent_coef {a[0]:.3f})')
PY
  rm -rf "$PROJ/logs/smoke_cap$CAP"
done
echo '== exp_id 9개 (있으면 같은 명령이 checkpoint resume; cap이 다른 checkpoint는 fingerprint가 거부)'
for E in square_tent12i_cap03_s1 square_tent12i_cap03_s2 square_tent12i_cap03_s3 square_tent12i_cap1_s1 square_tent12i_cap1_s2 square_tent12i_cap1_s3 can_tent12i_cap03_s1 can_tent12i_cap03_s2 can_tent12i_cap03_s3; do
  if [ -f "$PROJ/logs/$E.out" ]; then echo "$E: 이미 있음 -> $(grep '\[done\]\|\[eval\]' $PROJ/logs/$E.out | tail -n 1 | cut -c1-80)"; else echo "$E: 새로 시작"; fi
done
for G in square_tent12 square_tent12_hq square_baseline square_tent12i can_tent12i can_tent12i_hq; do echo -n "비교군 $G: "; ls -d $PROJ/logs/${G}_s* 2>/dev/null | wc -l; done
echo "processes: $(pgrep -fc '[t]rain_dsrl.py' || true)"; free -g | head -2; df -h /content | tail -n 1

## 7. 9개 시작 — `square_tent12i_cap03` × 3 + `square_tent12i_cap1` × 3 + `can_tent12i_cap03` × 3 (150k, 5k 격자). 6번이 실패했으면 띄우지 않는다

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
git pull --ff-only origin o2o
grep -q 'critic_alpha_cap' o2o_utils.py || { echo 'critic_alpha_cap missing from o2o_utils.py'; exit 2; }
PROJ=/content/drive/MyDrive/dsrl_project
mkdir -p "$PROJ/logs"
CFG_SQ='--config-path=cfg/robomimic --config-name=dsrl_square.yaml'
CFG_CAN='--config-path=cfg/robomimic --config-name=dsrl_can.yaml'
COMMON="variant=baseline log_dir=$PROJ/logs train.total_env_steps=150000 offline_mix.mode=none load_offline_data=False"
T12I="train.ent_coef=auto_0.3 train.target_ent=12"
launch () {
  EXP=$1; CFG=$2; shift 2
  if pgrep -af '[t]rain_dsrl.py' | grep -Fq "exp_id=$EXP"; then echo "already running: $EXP"; return; fi
  nohup python train_dsrl.py $CFG exp_id=$EXP "$@" $COMMON > "$PROJ/logs/$EXP.out" 2>&1 &
  echo "started $EXP (pid $!)"
}
for S in 1 2 3; do
  launch square_tent12i_cap03_s$S "$CFG_SQ"  seed=$S $T12I train.critic_alpha_cap=0.3
  launch square_tent12i_cap1_s$S  "$CFG_SQ"  seed=$S $T12I train.critic_alpha_cap=1.0
  launch can_tent12i_cap03_s$S    "$CFG_CAN" seed=$S $T12I train.critic_alpha_cap=0.3
done

## 8. 3분 후 자동 확인 — 9개 running, ERR 없음, 인자에 `train.critic_alpha_cap=0.3`/`1.0`과 `auto_0.3 … target_ent=12`. 30분 뒤 다시 돌리면 train_log의 `ent_coef`(actor α)·`critic_ent_coef`(타깃 온도; cap이 물리면 = cap)·qw_mean도 찍힌다

In [ ]:
%%bash
sleep 180
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
python scripts/inspect_runs.py --proj "$PROJ" --only square_tent12i_cap03_s,square_tent12i_cap1_s,can_tent12i_cap03_s
for E in square_tent12i_cap03_s1 square_tent12i_cap03_s2 square_tent12i_cap03_s3 square_tent12i_cap1_s1 square_tent12i_cap1_s2 square_tent12i_cap1_s3 can_tent12i_cap03_s1 can_tent12i_cap03_s2 can_tent12i_cap03_s3; do
  echo "== $E: $(grep '\[budget\]\|\[eval\]\|Traceback\|Error' "$PROJ/logs/$E.out" | tail -n 2 | tr '\n' ' ' | cut -c1-160)"
  T=$PROJ/logs/$E/train_log.csv
  [ -f "$T" ] && awk -F, 'NR==1{for(i=1;i<=NF;i++)c[$i]=i} END{printf "   train_log last: env_steps=%s ent_coef=%s critic_ent_coef=%s logp_mean=%s qw_mean=%s mu=%s\n", $c["env_steps"], $c["ent_coef"], $c["critic_ent_coef"], $c["logp_mean"], $c["qw_mean"], $c["mu_absmean"]}' "$T"
done
echo '== processes (인자 확인)'; pgrep -af '[t]rain_dsrl.py' | sed 's/.*exp_id=/exp_id=/' | cut -c1-200 || true
free -g | head -2

## 9. Keepalive — 마지막 프로세스가 끝나면 자동 반납

8번에서 오류가 없을 때만 실행하고 이 셀을 계속 실행 상태로 둡니다.

In [ ]:
import subprocess, time
from pathlib import Path
EXPECTED = [f'square_tent12i_{kind}_s{s}' for kind in ('cap03', 'cap1') for s in (1, 2, 3)] + [f'can_tent12i_cap03_s{s}' for s in (1, 2, 3)]
LOGS = Path('/content/drive/MyDrive/dsrl_project/logs')
assert LOGS.is_dir(), 'Drive가 이 런타임에 마운트돼 있지 않음 — 0번(또는 2번) 셀 먼저'

def running():
    out = subprocess.run(['ps', '-eo', 'pid,args'], capture_output=True, text=True).stdout
    return [line.strip() for line in out.splitlines() if 'train_dsrl.py' in line and any(f'exp_id={e}' in line for e in EXPECTED)]

def last_event(exp):
    path = LOGS / f'{exp}.out'
    if not path.exists(): return 'NO .out'
    lines = path.read_text(errors='replace').splitlines()[-500:]
    for line in reversed(lines):
        if any(x in line for x in ('[eval]', '[done]', 'Traceback', 'Error')): return line[:100]
    return 'starting'

seen_running = False
while True:
    procs = running()
    events = {e: last_event(e) for e in EXPECTED}
    print(time.strftime('%H:%M'), f'running {len(procs)}/{len(EXPECTED)}', '|', ' | '.join(f'{e}: {v}' for e, v in events.items()), flush=True)
    if procs:
        seen_running = True
    elif seen_running or all(v.startswith('[done]') for v in events.values()):
        print('all VM SC runs stopped -> unassigning', flush=True)
        from google.colab import runtime
        runtime.unassign()
        break
    else:
        print('이 런타임에 실행 중인 run이 없고 [done]도 아님 -> 7번 셀이 안 돌았거나 다른 런타임입니다. 반납하지 않고 종료.', flush=True)
        break
    time.sleep(600)

## 10. 결과 zip (끝난 뒤, CPU 런타임 + 0번 Drive 마운트만으로 됨). 로컬에서는 **새 폴더**(예: `logs/bundle_<날짜>/`)에 풀고 — 옛 번들이 섞인 `~/Downloads/logs`는 쓰지 않는다 —
`python scripts/plot_results.py --logs <새폴더>/logs --out results/<날짜>/square_transfer --axes "square_cap=square_baseline,square_tent12,square_tent12_hq,square_tent12i,square_tent12i_cap03,square_tent12i_cap1;square_rs=square_baseline,square_tent12,square_fixalpha_03,square_mix_prefill,square_tent12i,square_tent12i_rs02,square_fixa015_rs02;can_cap=baseline,tent12i,tent12i_hq,tent12i_cap03"`

cap이 물린 구간은 train_log의 `critic_ent_coef == cap`으로 읽는다(`plot_results.py`의 진단 패널에는 아직 없음; `ent_coef` 패널과 cap 값을 겹쳐 본다).

In [ ]:
%%bash
cd /content/drive/MyDrive/dsrl_project
rm -f csv_bundle.zip
zip -qr csv_bundle.zip logs -i "logs/*/eval_log.csv" "logs/*/train_log.csv" "logs/*.csv" "logs/pretrain/*_log.csv"
ls -lh csv_bundle.zip